# Data Quality — Delete Checks Notebook

**Use this notebook to remove checks from the registry.**

You can:
- **Soft delete** (set `is_active = false`) — the check data stays in the table but won't be validated
- **Hard delete** — permanently remove the row

Soft delete is safer for auditing; hard delete is cleaner if you don't need historical records.

In [ ]:
# -- Configuration -------------------------------------------------------------
# Preferred: load shared settings from Fabric config notebook.
try:
    ip = get_ipython()
    if ip is None:
        raise RuntimeError("IPython runtime not available")
    ip.run_line_magic("run", "data_quality_config_notebook")
except Exception:
    # Fallback keeps the notebook runnable if shared config execution is unavailable.
    LAKEHOUSE_NAME = "MyLakehouse"
    SCHEMA_NAME = "data_quality"

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"

## Current Checks

Here are all the checks currently registered:

In [ ]:
from IPython.display import display

display(spark.sql(f"""
SELECT
    check_name,
    model_name,
    workspace_id,
    dataset_id,
    is_active,
    run_frequency,
    workspace_name,
    dataset_name
FROM {FULL_SCHEMA}.check_registry
ORDER BY check_name, model_name, workspace_id, dataset_id
""").toPandas())

## Delete Checks

Choose how you want to delete:

1. **Soft Delete (recommended):** Set `is_active = false` — check stays in the table but won't be validated
2. **Hard Delete:** Permanently remove the row from the table

Edit the cell below to specify which checks to delete. You can match by:
- **check_name only** — deletes all models for that check
- **check_name + workspace_id + dataset_id** — deletes only that exact model identity row

In [ ]:
# ── EDIT BELOW: List the checks to delete ───────────────────────────────────
#
#   Format: (check_name, workspace_id_or_None, dataset_id_or_None)
#   - If workspace_id and dataset_id are None, deletes ALL models for that check_name
#   - If workspace_id and dataset_id are provided, deletes only that exact identity row
#
to_delete = [
    ("Total Sales", "66666666-7777-8888-9999-000000000000", "55555555-6666-7777-8888-999999999999"),
    ("Total Customers", None, None),
]

DELETE_METHOD = "soft"  # Choose: "soft" (set is_active=false) or "hard" (permanently delete)
# ─────────────────────────────────────────────────────────────────────────────

deleted_count = 0

for check_name, workspace_id, dataset_id in to_delete:
    check_name_sql = check_name.replace("'", "''")
    workspace_id_sql = workspace_id.replace("'", "''") if workspace_id else None
    dataset_id_sql = dataset_id.replace("'", "''") if dataset_id else None

    # Validate selector shape
    if (workspace_id is None) ^ (dataset_id is None):
        raise ValueError(
            f"Invalid selector for {check_name}: provide both workspace_id and dataset_id, or both as None"
        )

    if DELETE_METHOD == "soft":
        if workspace_id is None and dataset_id is None:
            # Count active rows BEFORE updating so the reported number is accurate
            count = spark.sql(f"""
                SELECT COUNT(*) FROM {FULL_SCHEMA}.check_registry
                WHERE check_name = '{check_name_sql}' AND is_active = true
            """).collect()[0][0]
            spark.sql(f"""
                UPDATE {FULL_SCHEMA}.check_registry
                SET is_active = false
                WHERE check_name = '{check_name_sql}'
            """)
            deleted_count += count
            print(f"  ✓ Deactivated {count} row(s) for check: {check_name}")
        else:
            # Count active target rows before update for exact reporting
            count = spark.sql(f"""
                SELECT COUNT(*) FROM {FULL_SCHEMA}.check_registry
                WHERE check_name = '{check_name_sql}'
                  AND workspace_id = '{workspace_id_sql}'
                  AND dataset_id = '{dataset_id_sql}'
                  AND is_active = true
            """).collect()[0][0]
            spark.sql(f"""
                UPDATE {FULL_SCHEMA}.check_registry
                SET is_active = false
                WHERE check_name = '{check_name_sql}'
                  AND workspace_id = '{workspace_id_sql}'
                  AND dataset_id = '{dataset_id_sql}'
            """)
            deleted_count += count
            print(f"  ✓ Deactivated {check_name} → {workspace_id}/{dataset_id} ({count} row(s))")

    elif DELETE_METHOD == "hard":
        if workspace_id is None and dataset_id is None:
            # Count target rows before delete for exact reporting
            count = spark.sql(f"""
                SELECT COUNT(*) FROM {FULL_SCHEMA}.check_registry
                WHERE check_name = '{check_name_sql}'
            """).collect()[0][0]
            spark.sql(f"""
                DELETE FROM {FULL_SCHEMA}.check_registry
                WHERE check_name = '{check_name_sql}'
            """)
            print(f"  ✓ Deleted {count} row(s) for check: {check_name}")
        else:
            # Count target rows before delete for exact reporting
            count = spark.sql(f"""
                SELECT COUNT(*) FROM {FULL_SCHEMA}.check_registry
                WHERE check_name = '{check_name_sql}'
                  AND workspace_id = '{workspace_id_sql}'
                  AND dataset_id = '{dataset_id_sql}'
            """).collect()[0][0]
            spark.sql(f"""
                DELETE FROM {FULL_SCHEMA}.check_registry
                WHERE check_name = '{check_name_sql}'
                  AND workspace_id = '{workspace_id_sql}'
                  AND dataset_id = '{dataset_id_sql}'
            """)
            print(f"  ✓ Deleted {check_name} → {workspace_id}/{dataset_id} ({count} row(s))")
        deleted_count += count
    else:
        raise ValueError("DELETE_METHOD must be either 'soft' or 'hard'")

print(f"\n✓  Deletion complete ({DELETE_METHOD} delete) — {deleted_count} row(s) affected")


## Verify Remaining Checks

In [ ]:
from IPython.display import display

remaining = spark.sql(f"""
SELECT
    check_name,
    model_name,
    workspace_id,
    dataset_id,
    is_active,
    run_frequency,
    workspace_name,
    dataset_name
FROM {FULL_SCHEMA}.check_registry
WHERE is_active = true
ORDER BY check_name, model_name, workspace_id, dataset_id
""").toPandas()

if len(remaining) > 0:
    display(remaining)
    print(f"\n✓  {len(remaining)} active check(s) remain")
else:
    print("✓  No active checks remaining")